In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import os
import random
# import tensorflow_model_optimization as tfmot
from metrics_tracking import F1Score, plot_metrics
from sklearn.metrics import roc_auc_score

# Metrics Before Compression:

Test AUC: 0.8723267912864685 (most important)

Test Precision: 1.0

Test Recall: 0.5199999809265137

Test F1: 0.684210479259491

Test loss: 0.08323828130960464

Test accuracy: 0.9896759390830994

In [19]:
#load dataset for model evaluation
#import the datasets to test
def load_data():
    data = np.load("Preprocessed_Data/roads_canids_windows_200hz_3s.npz")
    # Access arrays by their keys
    X_train = data["X_train"]
    y_train = data["y_train"]
    X_test  = data["X_test"]
    y_test  = data["y_test"]
    X_train = X_train[..., :-1] #TEMP FIX REMOVE LATER - FIX PREPROCESSING TO GET RID OF STRING COLUMN
    X_test  = X_test[..., :-1] #TEMP FIX REMOVE LATER
    print(X_train.shape)
    print(y_train.shape)
    print(X_test.shape)
    print(y_test.shape)
    data.close()
    return X_train, y_train, X_test, y_test
X_train, y_train, X_test, y_test = load_data()

(11280, 600, 23)
(11280,)
(13948, 600, 23)
(13948,)


In [42]:
import numpy as np
attack_indices = np.where(y_test == 1)[0]
rand_idx = np.random.choice(attack_indices)

x_attack = X_test[rand_idx]
y_attack = y_test[rand_idx]
print(rand_idx, y_attack)
print(x_attack)

13232 1
[[2.95000000e+01 3.16185696e-02 0.00000000e+00 ... 1.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [2.95050000e+01 2.51017705e+02 2.49474563e+02 ... 1.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [2.95100000e+01 1.67795862e+00 3.45930630e+02 ... 1.00000000e+00
  0.00000000e+00 0.00000000e+00]
 ...
 [3.24850000e+01 2.05189800e+02 2.03928387e+02 ... 1.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [3.24900000e+01 6.37674645e+01 4.81893766e+03 ... 1.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [3.24950000e+01 7.40847005e-01 2.17688516e+01 ... 1.00000000e+00
  0.00000000e+00 0.00000000e+00]]


In [43]:

def create_balanced_samples_for_validation(X_test, y_test, per_class=3):
    attack_idx = np.where(y_test == 1)[0]
    normal_idx = np.where(y_test == 0)[0]
    chosen_attacks = random.sample(list(attack_idx), per_class)
    chosen_normals = random.sample(list(normal_idx), per_class)
    chosen = chosen_attacks + chosen_normals
    scale = 1.472893953
    zero  = 0
    for i, idx in enumerate(chosen):
        x = X_test[idx].reshape(1, 600, 23)  # (batch, time, features)
        y = int(y_test[idx])
        x_q = np.clip(np.round(x/scale + zero), -128, 127).astype(np.int8)
        x_flat = x_q.flatten()
        label_str = "attack" if y == 1 else "normal"
        filename = f"Preprocessed_Data/sample_{i}_{label_str}_label{y}.csv"
        np.savetxt(filename, x_flat, fmt="%d", delimiter=",")
        print("saved:", filename)
    return chosen  
create_balanced_samples_for_validation(X_test, y_test, per_class=3)

saved: Preprocessed_Data/sample_0_attack_label1.csv
saved: Preprocessed_Data/sample_1_attack_label1.csv
saved: Preprocessed_Data/sample_2_attack_label1.csv
saved: Preprocessed_Data/sample_3_normal_label0.csv
saved: Preprocessed_Data/sample_4_normal_label0.csv
saved: Preprocessed_Data/sample_5_normal_label0.csv


[np.int64(13910),
 np.int64(13324),
 np.int64(13366),
 np.int64(1530),
 np.int64(9264),
 np.int64(9756)]

In [71]:
keras_model = keras.models.load_model( #import model for quantization
    "best_ROAD_model128_F1Fixed.keras",
    compile=True,
    custom_objects={"F1Score": F1Score},
    safe_mode=False
)
keras_model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_4 (Conv1D)               │ (None, 593, 128)       │        23,680 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 593, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_4      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 72,453 (283.02 KB)

 Trainable params: 24,065 (94.00 KB)

 Non-trainable params: 256 (1.00 KB)

 Optimizer params: 48,132 (188.02 KB)

In [72]:
#code here on out borrowed from ECE528 Assignment3 (Question 2/3)
def get_gzipped_model_size(keras_model, fname):
  # It returns the size of the gzipped model in bytes.
  import os
  import zipfile
  import tempfile
  keras_model.save(fname)
  _, zipped_file = tempfile.mkstemp('.zip')
  with zipfile.ZipFile(zipped_file, 'w', compression=zipfile.ZIP_DEFLATED) as f:
    f.write(fname, arcname=os.path.basename(fname))
  return os.path.getsize(zipped_file)


gzipped_size = get_gzipped_model_size(keras_model, "best_ROAD_model128_F1Fixed.keras")
print("Gzipped uncompressed model size (bytes):", gzipped_size)

Gzipped uncompressed model size (bytes): 273358


In [73]:
TFLITE_FP32 = "road_128_cnn.tflite"
TFLITE_QUANT_DYNAMIC = "road_128_cnn_quant_dyn.tflite"

X_test = X_test.astype(np.float32)
print("x_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

x_test shape: (13948, 600, 23)
y_test shape: (13948,)


In [74]:
converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
tflite_model = converter.convert()
with open(TFLITE_FP32, "wb") as f:
    f.write(tflite_model)

Saved artifact at '/tmp/tmpipumh7ia'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 600, 23), dtype=tf.float32, name='input_layer_4')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  136932460168208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460169168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460168976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460172240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460169552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460171280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460171088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460171856: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [75]:
#cconvert to tflite with dynamic conversion
dynamic_converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
dynamic_converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant = converter.convert()
with open(TFLITE_QUANT_DYNAMIC, "wb") as f:
    f.write(tflite_quant)

Saved artifact at '/tmp/tmpo696s6j1'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 600, 23), dtype=tf.float32, name='input_layer_4')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  136932460168208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460169168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460168976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460172240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460169552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460171280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460171088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460171856: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [ ]:
def representative_dataset():
    indices = np.arange(len(X_train))
    np.random.shuffle(indices)
    for i in indices[:500]:  # 500 random windows
        x = X_train[i:i+1].astype(np.float32)
        yield [x]

TFLITE_INT8 = "road_128_cnn_INT8.tflite"
converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8] ## converter.target_spec.supported_types = [tf.int8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_int8 = converter.convert()
with open(TFLITE_INT8, "wb") as f:
    f.write(tflite_int8)

Saved artifact at '/tmp/tmpek2msxc1'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 600, 23), dtype=tf.float32, name='input_layer_4')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  136932460168208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460169168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460168976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460172240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460169552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460171280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460171088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136932460171856: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [85]:

def evaluate_tflite_model_roc(model_path, x_data, y_labels, clip_range=(-1.0, 1.0)):
    """
    Evaluate a TFLite model (FP32 or INT8) and return ROC-AUC.
    Automatically handles input/output quantization and clipping for INT8.
    """
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    input_details  = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    n = x_data.shape[0]
    y_probs = np.zeros(n, dtype=np.float32)
    input_dtype  = input_details[0]['dtype']
    output_dtype = output_details[0]['dtype']
    in_scale, in_zero = input_details[0]['quantization']
    out_scale, out_zero = output_details[0]['quantization']
    for i in range(n):
        sample = x_data[i:i+1].astype(np.float32)
        # Clip inputs to safe range
        sample = np.clip(sample, clip_range[0], clip_range[1])
        # Quantize if INT8 input
        if input_dtype == np.int8:
            sample = (sample / in_scale + in_zero).astype(np.int8)
        interpreter.set_tensor(input_details[0]['index'], sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]['index'])
        # Dequantize if INT8 output
        if output_dtype == np.int8:
            output = (output.astype(np.float32) - out_zero) * out_scale
        # Binary classifier: probability of "Attack"
        y_probs[i] = output[0][0]
    auc = roc_auc_score(y_labels, y_probs)
    return auc

In [87]:
auc_int8 = evaluate_tflite_model_roc("road_int8.tflite", X_test, y_test)
print(f"INT8 TFLite ROC-AUC: {auc_int8:.4f}")

INT8 TFLite ROC-AUC: 0.5000


In Summary: INT8 Quantization fails bc completely removes effectiveness of model.
